<a href="https://colab.research.google.com/github/Marysia-max/SRMiO/blob/main/Aplikacja_Demo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q ultralytics gradio opencv-python numpy gTTS


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 20.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 98.2/98.2 kB 5.9 MB/s eta 0:00:00


In [2]:
from ultralytics import YOLO
import gradio as gr
import numpy as np
import os
import tempfile
from gtts import gTTS


Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


In [3]:
MODEL_PATH = "./best.pt"

if not os.path.exists(MODEL_PATH):
    raise FileNotFoundError(
        "Nie znaleziono pliku best.pt. Skopiuj best.pt do katalogu z notatnikiem Aplikacja_Demo.ipynb."
    )

model = YOLO(MODEL_PATH)
names = model.names
print("Załadowano model. Klasy:", names)


Załadowano model. Klasy: {0: 'Tshirt', 1: 'dress', 2: 'jacket', 3: 'pants', 4: 'shirt', 5: 'short', 6: 'skirt', 7: 'sweater'}


In [4]:
def tts_pl(text: str) -> str:
    if not text or text.strip() == "":
        text = "Brak detekcji."

    tts = gTTS(text=text, lang="pl")
    tmp = tempfile.NamedTemporaryFile(delete=False, suffix=".mp3")
    tmp.close()
    tts.save(tmp.name)
    return tmp.name


In [5]:
def predict(image: np.ndarray):
    results = model(image, imgsz=640, verbose=False)[0]

    labels = []
    for box in results.boxes:
        cls_id = int(box.cls[0].item())
        labels.append(names[cls_id])

    annotated = results.plot()
    unique_labels = ", ".join(sorted(set(labels))) if labels else "Nic nie wykryto"

    # Tekst do odczytu (MOWA)
    speak_text = f"Wykryto: {unique_labels}" if labels else "Nic nie wykryto"
    audio_path = tts_pl(speak_text)

    return annotated, unique_labels, audio_path


In [ ]:
demo = gr.Interface(
    fn=predict,
    inputs=gr.Image(sources=["upload", "webcam"], type="numpy"),
    outputs=[
        gr.Image(label="Detekcje"),
        gr.Textbox(label="Rozpoznane ubrania"),
        gr.Audio(label="Odczyt głosowy (TTS)", type="filepath")
    ],
    title="Rozpoznawanie ubrań – YOLOv8 (DEMO)"
)

demo.launch(debug=True)


It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://e15ec515e1c7587237.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
